In [0]:
class Bronze():
    def __init__(self):
        import os
        self.base_data_dir = os.path.abspath("..")
        self.checkpoint_dir = ("/Volumes/spark_stream/default/streaming_data/checkpoints")

    def getSchema(self):
        return """InvoiceNumber string
                , CreatedTime bigint
                , StoreID string
                , PosID string
                , CashierID string
                , CustomerType string
                , CustomerCardNo string
                , TotalAmount double
                , NumberOfItems bigint
                , PaymentMethod string
                , TaxableAmount double
                , CGST double
                , SGST double
                , CESS double
                , DeliveryType string
                , DeliveryAddress struct<AddressLine string
                , City string
                , ContactNumber string
                , PinCode string
                , State string>
                , InvoiceLineItems array<struct<ItemCode string
                , ItemDescription string
                , ItemPrice double
                , ItemQty bigint
                , TotalValue double>>
            """

    def readInvoices(self):
        from pyspark.sql.functions import col
        return (
                    spark.readStream
                        .format("json")
                        .schema(self.getSchema())
                        .load(f"{self.base_data_dir}/data/invoices")
                        .withColumn('inputFile', col('_metadata.file_path'))
                )   

    def process(self):
           print(f"Starting Bronze Stream...", end='')
           invoicesDF = self.readInvoices()
           sQuery = (
                    invoicesDF.writeStream
                            .queryName("bronze-ingestion")
                            .format("delta")
                            .option("checkpointLocation", f"{self.checkpoint_dir}/invoices_bz")
                            .outputMode("append")
                            .trigger(availableNow=True)
                            .toTable("spark_stream.default.invoices_bz")
                )
           print("Done\n")
           return sQuery

In [0]:
class Gold():
    def __init__(self):
        import os
        self.base_data_dir = os.path.abspath("..")
        self.checkpoint_dir = ("/Volumes/spark_stream/default/streaming_data/checkpoints")
        
    def readBronze(self):
        return spark.readStream.table("spark_stream.default.invoices_bz")
    
    def createGoldTable(self):
        spark.sql("""
            CREATE TABLE IF NOT EXISTS spark_stream.default.customer_rewards (
                CustomerCardNo STRING,
                TotalAmount DOUBLE,
                TotalPoints DOUBLE
            )
            USING DELTA
        """)

    @staticmethod
    def upsertToRewards(rewards_df, batch_id):
        rewards_df.createOrReplaceTempView("customer_rewards_df_temp_view")

        merge_statement = """
            MERGE INTO spark_stream.default.customer_rewards AS s
            USING customer_rewards_df_temp_view AS t
            ON s.CustomerCardNo = t.CustomerCardNo

            WHEN MATCHED THEN
                UPDATE SET
                    TotalAmount = t.TotalAmount,
                    TotalPoints = t.TotalPoints

            WHEN NOT MATCHED THEN
                INSERT (CustomerCardNo, TotalAmount, TotalPoints)
                VALUES (t.CustomerCardNo, t.TotalAmount, t.TotalPoints)
        """

        rewards_df.sparkSession.sql(merge_statement)

    def getAggregates(self, invoices_df):
        from pyspark.sql.functions import sum, expr
        return (invoices_df.groupBy("CustomerCardNo")
                    .agg(sum("TotalAmount").alias("TotalAmount"),
                         sum(expr("TotalAmount*0.02")).alias("TotalPoints"))
        )

    def saveResults(self, results_df):
        print(f"\nStarting Gold Stream...", end='')
        return (results_df.writeStream
                    .queryName("gold-update")
                    .option("checkpointLocation", f"{self.checkpoint_dir}/customer_rewards")
                    .outputMode("update")
                    .trigger(availableNow=True)
                    .foreachBatch(Gold.upsertToRewards)
                    .start()
                )
        print("Done")

    def process(self):
        self.createGoldTable()
        invoices_df = self.readBronze()
        aggregate_df = self.getAggregates(invoices_df)
        sQuery = self.saveResults(aggregate_df)
        return sQuery